# TBO InventoryIQ — Inventory Availability & Stockout Forecasting
**Phase 4: Machine Learning Development & Offline Evaluation**

Forecasts future available inventory quantities and predicts impending stockout events with critical threshold recall.

---


In [ ]:
import sys
from pathlib import Path
BASE_DIR = Path('..').resolve()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from modeling.registry.task_registry import TaskRegistry
from modeling.registry.dataset_registry import DatasetRegistry
from modeling.target_builder import prepare_task_dataset
from modeling.pipeline import run_task_pipeline
from modeling.model_persistence import load_model_artifact

print('Environment and modeling modules loaded successfully.')


## 1. Task Configuration & Dataset Inspection
Retrieve registered task `inventory_forecasting` from `TaskRegistry` and load modeling dataset.


In [ ]:
registry = TaskRegistry()
task_cfg = registry.get('inventory_forecasting')
print(f'Task: {task_cfg.task_name} | Type: {task_cfg.task_type.value} | Target: {task_cfg.target_column}')
print(f'Validation Strategy: {task_cfg.validation_strategy.value} (splits={task_cfg.n_splits})')
print(f'Excluded Features ({len(task_cfg.excluded_columns)}): {task_cfg.excluded_columns}')

X, y, df_full = prepare_task_dataset(task_cfg)
print(f'Prepared dataset: X shape = {X.shape}, y shape = {y.shape}')


## 2. Execute Modeling Pipeline
Runs fold-fitted preprocessing, baselines, Random Forest, XGBoost, LightGBM, and hyperparameter tuning.


In [ ]:
# Execute pipeline using reusable framework module
pipeline_results = run_task_pipeline('inventory_forecasting', n_iter_tuning=2)
champion_model = pipeline_results['champion_model']
print(f'Champion Model: {champion_model}')
print(f'Selection Rationale: {pipeline_results["rationale"]}')


## 3. Model Benchmark Comparison
Comparison of candidate algorithms across cross-validation folds.


In [ ]:
comp_df = pipeline_results['comparison_df']
display_cols = [c for c in comp_df.columns if c not in ['task_name', 'total_duration_sec']]
print(comp_df[display_cols].to_string(index=False))


## 4. Final Offline Test Evaluation & Diagnostics
Evaluates the selected champion model on the untouched test holdout set.


In [ ]:
test_eval = pipeline_results['test_evaluation']
print('Test Holdout Metrics:')
for k, v in test_eval['metrics'].items():
    print(f'  {k:20s}: {v}')

artifact, meta = load_model_artifact('inventory_forecasting')
print(f'Successfully reloaded persisted champion artifact: {artifact.model_name}')


## 5. Stockout Detection Analysis
Evaluates stockout detection recall against the 5-unit safety stockout threshold.


In [ ]:
if 'stockout_diagnostics' in test_eval:
    print('Stockout Diagnostics:')
    for k, v in test_eval['stockout_diagnostics'].items():
        print(f'  {k:25s}: {v}')
